# 🩺 Diabetes HbA1c Prediction using AutoML

**🎯 Objective**: Predict post-treatment HbA1c levels using state-of-the-art machine learning on comprehensive diabetes intervention data.

**📊 Dataset**: 6,990+ participants across 3 intervention studies with 80+ clinical features including:
- Laboratory values (FBS, PPBS, HbA1c, lipid panels)
- Anthropometric measurements (BMI, waist-hip ratio, vital signs) 
- Demographics and lifestyle factors
- Family history and medical backgrounds

**🤖 Method**: H2O AutoML with automated algorithm comparison, hyperparameter optimization, and clinical validation metrics

**🎯 Target Performance**: R² > 0.85, RMSE < 0.5 HbA1c units, Clinical Accuracy > 80% for deployment readiness

**⏱️ Expected Runtime**: 15-20 minutes for complete pipeline execution (2-hour AutoML budget)

## 1. Environment Setup & Package Installation

Configure the runtime before executing the rest of the notebook. Complete the data access step, then install the dependencies so the remaining cells run without interruption.

## 📋 Quick Start Checklist

**Essential Setup Steps** (Execute in order):

1. **Environment**: Connect to your runtime (Colab GPU/CPU or local kernel)
2. **Data Access**: Enable Google Drive or prepare manual CSV uploads  
3. **Execute Cells**: Run cells sequentially starting with:
   - **Cell 4**: Package Installation 
   - **Cell 6**: Library Imports
   - **Cell 8**: Configuration Setup
   - **Cell 10**: Data Access Setup
   - Continue through data loading, modeling, and reporting sections

**🎯 Expected Runtime**: 15-20 minutes for complete pipeline execution

In [ ]:
# =============================================================================
# PACKAGE INSTALLATION FOR GOOGLE COLAB
# =============================================================================

import subprocess
import sys
from typing import Optional, Sequence, Tuple

INSTALL_GROUPS: Sequence[Tuple[str, Sequence[Tuple[str, Optional[str]]]]] = (
    (
        "Core AutoML Libraries",
        (
            ("h2o", ">=3.44.0.1"),
            ("scikit-learn", ">=1.4.0"),
            ("xgboost", ">=1.7.0"),
            ("lightgbm", ">=3.3.0"),
        ),
    ),
    (
        "Data Science",
        (
            ("pandas", ">=2.1.0"),
            ("numpy", ">=1.24.0"),
            ("scipy", ">=1.10.0"),
            ("statsmodels", ">=0.14.0"),
        ),
    ),
    (
        "Visualization",
        (
            ("matplotlib", ">=3.7.0"),
            ("seaborn", ">=0.12.0"),
            ("plotly", ">=5.20.0"),
            ("kaleido", ">=0.2.1"),
        ),
    ),
    (
        "Interpretability & Medical Analytics",
        (
            ("shap", ">=0.42.0"),
            ("lime", ">=0.2.0"),
            ("eli5", ">=0.13.0"),
            ("lifelines", ">=0.27.0"),
            ("pingouin", ">=0.5.0"),
        ),
    ),
)


def install_package(package: str, constraint: Optional[str] = None, upgrade: bool = False) -> None:
    """Install a Python package with optional version constraint and graceful fallback."""
    spec = f"{package}{constraint or ''}"
    cmd = [sys.executable, "-m", "pip", "install"]
    if upgrade:
        cmd.append("--upgrade")
    cmd.append(spec)

    try:
        subprocess.run(cmd, check=True, capture_output=True, text=True)
        print(f"✅ {spec}")
    except subprocess.CalledProcessError as exc:
        print(f"❌ {spec} failed: {exc.stderr.strip().splitlines()[-1] if exc.stderr else 'Unknown error'}")
        fallback_cmd = [sys.executable, "-m", "pip", "install", spec, "--no-deps"]
        try:
            subprocess.run(fallback_cmd, check=True, capture_output=True, text=True)
            print(f"✅ {spec} (installed without dependencies)")
        except subprocess.CalledProcessError:
            print(f"⚠️  Skipped {spec}. Resolve manually if required.")


print("📦 Installing required packages (rerun only if versions change)...")
for group_name, packages in INSTALL_GROUPS:
    print(f"\n— {group_name} —")
    for package, constraint in packages:
        install_package(package, constraint)

print("\n🎉 Package installation completed. Restart the runtime if prompted by Colab.")

## 3. Core Library Imports
Run the following cell **after** package installation to import all libraries used throughout the workflow and apply consistent plotting styles.

In [ ]:
# =============================================================================
# CORE IMPORTS & GLOBAL SETTINGS
# =============================================================================

import os
import time
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import h2o
from h2o.automl import H2OAutoML  # Added missing H2OAutoML import
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score  # Added missing metrics

# Global plotting style for consistency
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

print("✅ Core libraries imported successfully.")

## 4. Notebook Configuration
Update the configuration dictionary below to control dataset paths, feature filtering, and AutoML runtime budgets before running the pipeline.

In [ ]:
# =============================================================================
# NOTEBOOK CONFIGURATION
# =============================================================================

NOTEBOOK_CONFIG: Dict[str, object] = {
    "target_column": "PostBLHBA1C",
    "dataset_files": {
        "dataset1": "nmbfinalDiabetes (4)_selected_columns_cleaned_processed_final_imputed.csv",
        "dataset2": "nmbfinalnewDiabetes (3)_selected_columns_cleaned_processed_final_imputed.csv",
        "dataset3": "PrePostFinal (3)_selected_columns_cleaned_processed_final_imputed.csv",
    },
    "feature_filters": {
        "drop_duplicate_columns": True,
        "drop_constant_columns": True,
        "high_missing_fraction": 0.98,
        "exclude_columns": [],
        "protected_columns": ["dataset_source"],
    },
    "reporting_thresholds": {
        "moderate_missing": 20,
    },
    "clinical_thresholds": {
        "normal_upper": 5.7,
        "prediabetes_upper": 6.5,
        "clinical_accuracy_tolerance": 0.5,
        "clinical_accuracy_secondary_tolerance": 1.0,
    },
    "evaluation_thresholds": {
        "excellent_rmse": 0.5,
        "good_rmse": 1.0,
        "excellent_r2": 0.85,
        "good_r2": 0.75,
        "clinical_accuracy_target": 80.0,
        "clinical_accuracy_watch": 70.0,
    },
    "modeling": {
        "identifier_columns": ["dataset_source"],
    },
    "train_test_split": {
        "test_size": 0.2,
        "validation_fraction": 0.2,
        "random_state": 42,
        "stratify_bins": 5,
    },
    "automl": {
        "max_models": 40,
        "max_runtime_secs": 2 * 60 * 60,
        "seed": 42,
        "nfolds": 5,
        "sort_metric": "RMSE",
        "stopping_tolerance": 0.001,
        "stopping_rounds": 3,
        "project_name": "diabetes_hba1c_automl",
    },
}

print("Configuration loaded. Adjust NOTEBOOK_CONFIG if different runtime or filtering is required.")

## 5. Data Access & Workspace Paths (Colab-Friendly)
Mount Google Drive or define an explicit data directory before loading the datasets. This cell auto-detects Colab and falls back to local execution when appropriate.

In [ ]:
# =============================================================================
# GOOGLE COLAB DATA SETUP (OPTIONAL)
# =============================================================================

RUNNING_IN_COLAB = False
DEFAULT_RELATIVE_DATA_DIR = "final_imputed_data"

try:  # Detect Colab
    import google.colab  # type: ignore
    RUNNING_IN_COLAB = True
except ImportError:
    RUNNING_IN_COLAB = False

if RUNNING_IN_COLAB:
    print("📌 Detected Google Colab environment")
    from google.colab import drive  # type: ignore
    if not Path("/content/drive").exists() or not any(Path("/content/drive").iterdir()):
        print("🔗 Mounting Google Drive (update permissions when prompted)...")
        drive.mount("/content/drive")
    base_dir = Path(os.environ.get("DIABETES_PROJECT_BASE_DIR", "/content/drive/MyDrive/poornima-ML"))
else:
    base_dir = Path(os.environ.get("DIABETES_PROJECT_BASE_DIR", ".")).resolve()
    print("💻 Running locally. Override DIABETES_PROJECT_BASE_DIR if needed.")

# Allow explicit override for data folder
explicit_data_dir = os.environ.get("DIABETES_DATA_DIR")
if explicit_data_dir:
    data_dir = Path(explicit_data_dir)
else:
    data_dir = base_dir if base_dir.name == DEFAULT_RELATIVE_DATA_DIR else base_dir / DEFAULT_RELATIVE_DATA_DIR

DATA_DIR = data_dir.resolve()
print(f"📁 DATA_DIR set to: {DATA_DIR}")

if not DATA_DIR.exists():
    print("⚠️  DATA_DIR does not exist yet. Ensure the three CSV files are placed here before running the next sections.")

## 6. Optional: Manual CSV Upload (Colab)
Run the next cell if you prefer to upload the three CSV files from your local computer into the Colab session. The files will be saved under `DATA_DIR`.

In [ ]:
if RUNNING_IN_COLAB and not DATA_DIR.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    from google.colab import files  # type: ignore
    print("⬆️  Select the three CSV files generated by the preprocessing pipeline.")
    uploads = files.upload()
    for filename, data in uploads.items():
        target_path = DATA_DIR / filename
        with open(target_path, "wb") as fout:
            fout.write(data)
        print(f"✅ Saved {filename} to {target_path}")
else:
    print("ℹ️  Manual upload skipped (DATA_DIR already exists or running locally).")

## 7. Data Health Utility Functions
The helper below profiles missingness, duplicates, and identifier-like columns. It is invoked automatically after data loading.

In [ ]:
# =============================================================================
# DATA HEALTH CHECKS & SUMMARY STATS
# =============================================================================

def summarize_dataset_health(df: pd.DataFrame, config: Dict[str, object]) -> Dict[str, object]:
    """Profile dataset readiness for modelling with missingness, stats, and target checks."""

    print("\n🩺 DATASET HEALTH CHECK")
    print("=" * 60)
    print(f"👥 Samples: {len(df):,}")
    print(f"📈 Features: {len(df.columns):,}")

    missing_info = (
        df.isnull()
        .mean()
        .mul(100)
        .sort_values(ascending=False)
        .rename("missing_pct")
    )

    severe_missing = missing_info[missing_info >= float(config['feature_filters']['high_missing_fraction']) * 100]
    moderate_threshold = config.get('reporting_thresholds', {}).get('moderate_missing', 20)
    moderate_missing = missing_info[(missing_info >= moderate_threshold) & (~missing_info.index.isin(severe_missing.index))]

    print("\n🧪 Missingness Overview:")
    print(f"   • Columns ≥{float(config['feature_filters']['high_missing_fraction']) * 100:.0f}% missing: {len(severe_missing)}")
    print(f"   • Columns ≥{moderate_threshold}% missing: {len(moderate_missing)}")

    if not severe_missing.empty:
        print("\n   ⚠️  Severe missingness columns:")
        for col, pct in severe_missing.items():
            print(f"      - {col}: {pct:.1f}%")

    if not moderate_missing.empty:
        print("\n   ⚠️  Moderate missingness columns:")
        for col, pct in moderate_missing.head(10).items():
            print(f"      - {col}: {pct:.1f}%")
        if len(moderate_missing) > 10:
            print(f"      ... +{len(moderate_missing) - 10} more")

    numeric_stats = df.select_dtypes(include=[np.number]).describe().T
    numeric_stats['missing_pct'] = df.select_dtypes(include=[np.number]).isnull().mean().mul(100)

    print("\n📊 Numeric Feature Summary (first 5):")
    display(numeric_stats.head())

    target_col = config['target_column']
    if target_col in df.columns:
        target = df[target_col]
        print(f"\n🎯 Target Variable ({target_col}):")
        print(f"   Non-missing: {target.notna().sum():,}")
        print(f"   Mean/Median: {target.mean():.2f} / {target.median():.2f}")
        print(f"   Std Dev: {target.std():.2f}")
    else:
        print(f"\n⚠️  Target column '{target_col}' not found!")

    duplicated_cols = df.columns[df.columns.duplicated()].tolist()
    if duplicated_cols:
        print("\n⚠️  Duplicated columns detected:")
        for col in duplicated_cols:
            print(f"   - {col}")
    else:
        print("\n✅ No duplicated columns detected.")

    constant_cols = [col for col in df.columns if df[col].nunique(dropna=True) <= 1]
    if constant_cols:
        print("\n⚠️  Constant columns detected:")
        for col in constant_cols[:10]:
            print(f"   - {col}")
        if len(constant_cols) > 10:
            print(f"   ... +{len(constant_cols) - 10} more")

    identifier_keywords = config.get('identifier_keywords', ['ID', 'PATIENT', 'SUBJECT', 'SOURCE'])
    id_candidates = [col for col in df.columns if any(keyword in col.upper() for keyword in identifier_keywords)]
    if id_candidates:
        print("\n🆔 Potential identifier columns:")
        for col in id_candidates:
            print(f"   - {col}")
    else:
        print("\n✅ No identifier-like columns detected.")

    return {
        "missing_overview": missing_info,
        "numeric_stats": numeric_stats,
        "constant_columns": constant_cols,
        "identifier_candidates": id_candidates,
    }

print("ℹ️ Call `summarize_dataset_health(df_main, NOTEBOOK_CONFIG)` after data loading to review readiness.")

## 8. Data Loading & Quality Validation
Load and combine three diabetes intervention datasets with comprehensive quality validation.

In [ ]:
# =============================================================================
# DATASET LOADING WITH QUALITY CONTROLS
# =============================================================================

def load_diabetes_datasets(
    data_dir: Path = DATA_DIR,
    dataset_mapping: Optional[Dict[str, str]] = None,
) -> Dict[str, pd.DataFrame]:
    """Load the three curated datasets and return a dictionary keyed by dataset name."""

    dataset_files = dataset_mapping or NOTEBOOK_CONFIG.get("dataset_files", {})
    if not dataset_files:
        raise ValueError("NOTEBOOK_CONFIG['dataset_files'] must be populated with at least one CSV filename.")

    print("📂 Loading Diabetes Intervention Datasets...")
    print(f"📁 Data directory: {data_dir}")

    datasets: Dict[str, pd.DataFrame] = {}

    for alias, filename in dataset_files.items():
        filepath = data_dir / filename
        if not filepath.exists():
            raise FileNotFoundError(f"Dataset file not found: {filepath}")

        df = pd.read_csv(
            filepath,
            low_memory=False,
            na_values=["", "NA", "null", "NULL", "NaN"],
        )
        df["dataset_source"] = alias
        datasets[alias] = df

        print(f"✅ {alias}: {df.shape[0]:,} rows × {df.shape[1]:,} columns | File: {filename}")

    combined_df = pd.concat(datasets.values(), ignore_index=True, sort=False)
    datasets["combined_raw"] = combined_df

    print(f"\n📎 Combined dataset: {combined_df.shape[0]:,} rows × {combined_df.shape[1]:,} columns")
    return datasets


def apply_feature_filters(
    df: pd.DataFrame,
    target_column: str,
    filters: Dict[str, object],
) -> Tuple[pd.DataFrame, Dict[str, List[str]]]:
    """Apply configurable column filters and return the refined dataset with a summary report."""

    protected = set(filters.get("protected_columns", [])) | {target_column}
    drop_columns = set(filters.get("exclude_columns", [])) - protected
    report: Dict[str, List[str]] = {}

    if filters.get("drop_duplicate_columns", True):
        duplicate_mask = df.T.duplicated()
        duplicate_cols = df.columns[duplicate_mask].tolist()
        report["duplicate_columns"] = duplicate_cols
        drop_columns |= set(duplicate_cols)

    if filters.get("drop_constant_columns", True):
        constant_cols = [col for col in df.columns if df[col].nunique(dropna=False) <= 1]
        report["constant_columns"] = constant_cols
        drop_columns |= set(constant_cols)

    high_missing_fraction = float(filters.get("high_missing_fraction", 1.0))
    if high_missing_fraction < 1.0:
        high_missing_cols = [col for col in df.columns if df[col].isna().mean() >= high_missing_fraction]
        report["high_missing_columns"] = high_missing_cols
        drop_columns |= set(high_missing_cols)

    drop_columns -= protected
    refined_df = df.drop(columns=list(drop_columns), errors="ignore")

    report["dropped_columns"] = sorted(drop_columns)
    report["retained_columns"] = sorted(refined_df.columns.tolist())
    return refined_df, report


def validate_data_quality(df: pd.DataFrame, dataset_name: str = "Dataset") -> None:
    """Comprehensive data quality validation for medical datasets."""

    print(f"\n🔍 Data Quality Validation: {dataset_name}")
    print("=" * 60)
    print(f"📊 Shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")

    missing_count = df.isnull().sum().sum()
    print(f"❓ Missing values: {missing_count:,} ({missing_count / df.size * 100:.2f}%)")

    target_column = NOTEBOOK_CONFIG["target_column"]
    if target_column in df.columns:
        target = df[target_column]
        print(f"\n🎯 Target ({target_column}) distribution:")
        print(f"   Range: {target.min():.2f} – {target.max():.2f} %")
        print(f"   Mean ± SD: {target.mean():.2f} ± {target.std():.2f} %")
        print(f"   Median (IQR): {target.median():.2f} ({target.quantile(0.25):.2f} – {target.quantile(0.75):.2f}) %")

        normal = (target < 5.7).sum()
        prediabetes = ((target >= 5.7) & (target < 6.5)).sum()
        diabetes = (target >= 6.5).sum()
        total = len(target)

        print(f"\n🏥 Clinical categories:")
        print(f"   Normal (<5.7%): {normal:,} ({normal / total * 100:.1f}%)")
        print(f"   Prediabetes (5.7–6.4%): {prediabetes:,} ({prediabetes / total * 100:.1f}%)")
        print(f"   Diabetes (≥6.5%): {diabetes:,} ({diabetes / total * 100:.1f}%)")

    dtype_counts = df.dtypes.value_counts()
    print("\n📋 Column data types:")
    for dtype, count in dtype_counts.items():
        print(f"   {dtype}: {count} columns")

    memory_mb = df.memory_usage(deep=True).sum() / 1024 ** 2
    print(f"\n💾 Memory usage: {memory_mb:.1f} MB")
    print("✅ Data quality validation completed")


# Load datasets with configured filters
try:
    raw_datasets = load_diabetes_datasets(DATA_DIR, NOTEBOOK_CONFIG.get("dataset_files"))
    combined_raw = raw_datasets["combined_raw"].copy()

    refined_df, filter_report = apply_feature_filters(
        combined_raw,
        NOTEBOOK_CONFIG["target_column"],
        NOTEBOOK_CONFIG.get("feature_filters", {}),
    )

    print("\n🧹 Feature filtering summary:")
    for key, values in filter_report.items():
        if key in {"retained_columns"}:
            continue
        if values:
            print(f"   {key.replace('_', ' ').title()}: {len(values)}")
    print(f"   Retained columns: {len(filter_report['retained_columns'])}")

    validate_data_quality(refined_df, "Combined Dataset (Filtered)")
    df_main = refined_df.copy()

    print("\n📋 Running dataset health summary...")
    dataset_health = summarize_dataset_health(df_main, NOTEBOOK_CONFIG)

except Exception as exc:
    print(f"❌ Critical error in data loading: {exc}")
    raise

In [ ]:
# -- Quick EDA (Optional) --
if 'df_main' in locals():
    target_col = NOTEBOOK_CONFIG['target_column']
    import matplotlib.pyplot as plt
    import seaborn as sns

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.histplot(df_main[target_col], bins=30, kde=True, ax=axes[0], color='steelblue')
    axes[0].axvline(5.7, color='orange', linestyle='--', label='Prediabetes threshold')
    axes[0].axvline(6.5, color='red', linestyle='--', label='Diabetes threshold')
    axes[0].set_title(f'{target_col} Distribution')
    axes[0].set_xlabel('HbA1c (%)')
    axes[0].legend()

    numeric_cols = df_main.select_dtypes(include=[np.number]).columns.tolist()
    corr = df_main[numeric_cols].corr()[target_col].drop(target_col).abs().sort_values(ascending=False).head(10)
    sns.barplot(x=corr.values, y=corr.index, ax=axes[1], palette='viridis')
    axes[1].set_title(f'Top 10 Absolute Correlations with {target_col}')
    axes[1].set_xlabel('Absolute Correlation')

    plt.tight_layout()
    plt.show()

## 9. Exploratory Data Analysis (EDA)

Comprehensive medical data exploration with clinical insights and statistical validation.

In [ ]:
# =============================================================================
# TARGET VARIABLE ANALYSIS & CLINICAL INSIGHTS
# =============================================================================

def analyze_target_distribution(df: pd.DataFrame, target_col: str) -> Dict[str, object]:
    """Summarise target behaviour with clinical thresholds and statistical diagnostics."""

    print(f"🎯 TARGET VARIABLE ANALYSIS: {target_col}")
    print("=" * 60)

    target = df[target_col].dropna()
    print("\n📊 Statistical Summary:")
    print(f"   Sample size: {len(target):,}")
    print(f"   Range: {target.min():.2f} – {target.max():.2f}%")
    print(f"   Mean ± SD: {target.mean():.2f} ± {target.std():.2f}%")
    print(f"   Median (IQR): {target.median():.2f} ({target.quantile(0.25):.2f} – {target.quantile(0.75):.2f})%")
    print(f"   Skewness: {stats.skew(target):.3f}")
    print(f"   Kurtosis: {stats.kurtosis(target):.3f}")

    print("\n🏥 Clinical Category Distribution:")
    categories = ['Normal (<5.7%)', 'Prediabetes (5.7–6.4%)', 'Diabetes (≥6.5%)']
    thresholds = [(None, 5.7), (5.7, 6.5), (6.5, None)]
    counts, percentages = [], []
    for lower, upper in thresholds:
        if lower is None:
            mask = target < upper
        elif upper is None:
            mask = target >= lower
        else:
            mask = (target >= lower) & (target < upper)
        counts.append(mask.sum())
        percentages.append(mask.mean() * 100)

    for cat, count, pct in zip(categories, counts, percentages):
        print(f"   {cat}: {count:,} ({pct:.1f}%)")

    q1, q3 = target.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers_low = (target < lower_bound).sum()
    outliers_high = (target > upper_bound).sum()

    print("\n📊 Outlier Analysis (1.5×IQR Method):")
    print(f"   Lower bound: {lower_bound:.2f}% | High bound: {upper_bound:.2f}%")
    print(f"   Low outliers: {outliers_low:,} | High outliers: {outliers_high:,}")
    print(f"   Total outliers: {outliers_low + outliers_high:,} ({(outliers_low + outliers_high) / len(target) * 100:.1f}%)")

    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(f'{target_col} Distribution Analysis', fontsize=16, fontweight='bold')

    axes[0, 0].hist(target, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
    axes[0, 0].axvline(5.7, color='orange', linestyle='--', linewidth=2, label='Prediabetes threshold')
    axes[0, 0].axvline(6.5, color='red', linestyle='--', linewidth=2, label='Diabetes threshold')
    axes[0, 0].set_xlabel('HbA1c (%)')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].set_title('Distribution with Clinical Thresholds')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    box_plot = axes[0, 1].boxplot(target, patch_artist=True)
    box_plot['boxes'][0].set_facecolor('lightcoral')
    axes[0, 1].set_ylabel('HbA1c (%)')
    axes[0, 1].set_title('Box Plot with Outliers')
    axes[0, 1].grid(True, alpha=0.3)

    stats.probplot(target, dist='norm', plot=axes[1, 0])
    axes[1, 0].set_title('Q-Q Plot (Normality Check)')
    axes[1, 0].grid(True, alpha=0.3)

    colors = ['lightgreen', 'gold', 'lightcoral']
    axes[1, 1].pie(counts, labels=categories, colors=colors, autopct='%1.1f%%', startangle=90)
    axes[1, 1].set_title('Clinical Category Distribution')

    plt.tight_layout()
    plt.show()

    print("\n🧪 Normality Tests:")
    sample = target.sample(min(5000, len(target)), random_state=42)
    shapiro_stat, shapiro_p = stats.shapiro(sample)
    ks_stat, ks_p = stats.kstest(sample, 'norm', args=(sample.mean(), sample.std()))
    print(f"   Shapiro–Wilk p-value: {shapiro_p:.2e} ({'approx. normal' if shapiro_p > 0.05 else 'non-normal'})")
    print(f"   Kolmogorov–Smirnov p-value: {ks_p:.2e} ({'approx. normal' if ks_p > 0.05 else 'non-normal'})")

    return {
        "statistics": target.describe(),
        "clinical_distribution": dict(zip(categories, counts)),
        "outliers": {"low": outliers_low, "high": outliers_high},
        "normality_tests": {"shapiro_p": shapiro_p, "ks_p": ks_p},
    }


def analyze_feature_categories(df: pd.DataFrame, target_col: str) -> Dict[str, List[str]]:
    """Categorise clinical variables into logical groups for downstream analysis."""

    print("\n🔬 FEATURE CATEGORISATION ANALYSIS")
    print("=" * 60)

    columns = df.columns.tolist()
    pattern_lookup = {
        'clinical_labs': ['FBS', 'PPBS', 'HBA1C', 'CHOLESTEROL', 'TRIGLYCERIDES', 'LDL'],
        'vital_signs': ['SYSTOLIC', 'DIASTOLIC', 'HEIGHT', 'WEIGHT', 'HIP', 'WAIST'],
        'demographics': ['AGE', 'GENDER', 'AREA', 'MARITAL', 'EDUCATION', 'OCCUPATION'],
        'family_history': ['DIAFATHER', 'DIAMOTHER', 'DIABROTHER', 'DIASISTER'],
        'lifestyle': ['TOB', 'ALCOHOL', 'SLEEP', 'ACTIVITY', 'BREAKFAST', 'FIBER', 'FRUIT', 'VEGETABLE'],
        'derived': ['DURATION', 'STATUS', 'CURRENT_', 'OPTIMAL', 'RISK', 'SCORE'],
        'identifiers': ['SOURCE', 'GROUP', 'ID'],
    }

    feature_categories: Dict[str, List[str]] = {"target": [target_col]}
    for label, keywords in pattern_lookup.items():
        feature_categories[label] = [col for col in columns if any(keyword in col.upper() for keyword in keywords)]

    categorized = set().union(*feature_categories.values())
    uncategorised = [col for col in columns if col not in categorized]
    feature_categories['uncategorised'] = uncategorised

    for category, features in feature_categories.items():
        print(f"\n📋 {category.replace('_', ' ').title()} ({len(features)} features)")
        for feature in features[:5]:
            print(f"   • {feature}")
        if len(features) > 5:
            print(f"   ... +{len(features) - 5} more")

    total = len(columns)
    categorised_count = total - len(uncategorised)
    print("\n📊 CATEGORISATION SUMMARY")
    print(f"   Total features: {total}")
    print(f"   Categorised: {categorised_count} ({categorised_count / total * 100:.1f}%)")
    print(f"   Uncategorized: {len(uncategorised)} ({len(uncategorised) / total * 100:.1f}%)")

    return feature_categories


print("🚀 Starting Comprehensive Exploratory Data Analysis...")
print(f"📊 Analyzing dataset with {len(df_main):,} samples and {len(df_main.columns):,} features")

target_column = NOTEBOOK_CONFIG['target_column']
target_analysis = analyze_target_distribution(df_main, target_column)
feature_categories = analyze_feature_categories(df_main, target_column)

print("\n✅ Exploratory Data Analysis completed successfully!")

## 10. Feature Engineering & Clinical Feature Enhancement

Creating medically-relevant derived features and optimizing the feature set for AutoML.

In [ ]:
# =============================================================================
# CLINICAL FEATURE ENGINEERING PIPELINE
# =============================================================================

def create_clinical_features(df):
    """
    Create medically-relevant derived features for diabetes prediction
    
    This function creates evidence-based clinical features including:
    - Body Mass Index (BMI) and categories
    - Blood pressure risk categories  
    - Waist-to-hip ratio (WHR) for central obesity assessment
    - Metabolic syndrome component scores
    - Lifestyle risk aggregation
    - Family history composite scores
    
    Args:
        df (pd.DataFrame): Input dataset with clinical measurements
        
    Returns:
        Tuple[pd.DataFrame, List[str]]: Enhanced dataset and list of new feature names
    """
    
    print("⚙️ CLINICAL FEATURE ENGINEERING")
    print("=" * 40)
    
    df_enhanced = df.copy()
    features_created = []
    
    # 1. Body Mass Index (BMI) - Critical diabetes risk factor
    if 'PreRheight' in df.columns and 'PreRweight' in df.columns:
        print("📊 Creating BMI features...")
        
        # Validate height and weight ranges (basic sanity checks)
        valid_height = (df_enhanced['PreRheight'] >= 100) & (df_enhanced['PreRheight'] <= 250)  # cm
        valid_weight = (df_enhanced['PreRweight'] >= 30) & (df_enhanced['PreRweight'] <= 300)   # kg
        
        # Convert height from cm to meters and calculate BMI
        height_m = df_enhanced['PreRheight'] / 100
        df_enhanced['BMI'] = np.where(
            valid_height & valid_weight,
            df_enhanced['PreRweight'] / (height_m ** 2),
            np.nan
        )
        
        # BMI Categories (WHO classification for Asian populations)
        df_enhanced['BMI_Category'] = pd.cut(
            df_enhanced['BMI'], 
            bins=[0, 18.5, 23, 25, 30, 100],  # Asian-specific cutoffs
            labels=['Underweight', 'Normal', 'Overweight', 'Obese_I', 'Obese_II+'],
            include_lowest=True
        )
        
        features_created.extend(['BMI', 'BMI_Category'])
        print(f"   ✅ BMI: Valid calculations for {df_enhanced['BMI'].notna().sum():,} samples")
        print(f"   📋 BMI range: {df_enhanced['BMI'].min():.1f} - {df_enhanced['BMI'].max():.1f} kg/m²")
    else:
        print("   ⚠️  Height/weight columns not found - skipping BMI calculation")
    
    # 2. Blood Pressure Categories (AHA/ESC Guidelines)
    if 'PreRsystolicfirst' in df.columns and 'PreRdiastolicfirst' in df.columns:
        print("🩺 Creating blood pressure categories...")
        
        def categorize_bp(systolic, diastolic):
            """Categorize BP according to AHA/ESC 2018 guidelines"""
            if pd.isna(systolic) or pd.isna(diastolic):
                return 'Unknown'
            # Validate BP ranges (basic sanity checks)
            if systolic < 70 or systolic > 250 or diastolic < 40 or diastolic > 150:
                return 'Invalid'
            elif systolic < 120 and diastolic < 80:
                return 'Normal'
            elif systolic < 130 and diastolic < 80:
                return 'Elevated'
            elif (systolic >= 130 and systolic < 140) or (diastolic >= 80 and diastolic < 90):
                return 'Stage_1_HTN'
            elif systolic >= 140 or diastolic >= 90:
                return 'Stage_2_HTN'
            else:
                return 'Unknown'
        
        df_enhanced['BP_Category'] = df_enhanced.apply(
            lambda row: categorize_bp(row['PreRsystolicfirst'], row['PreRdiastolicfirst']), axis=1
        )
        
        features_created.append('BP_Category')
        bp_counts = df_enhanced['BP_Category'].value_counts()
        print(f"   ✅ BP Categories: {bp_counts.to_dict()}")
    else:
        print("   ⚠️  Blood pressure columns not found - skipping BP categorization")
    
    # 3. Waist-to-Hip Ratio (WHR) - Central obesity indicator
    if 'PreRwaist' in df.columns and 'PreRhip' in df.columns:
        print("📏 Creating waist-to-hip ratio features...")
        
        # Validate measurements (basic sanity checks)
        valid_waist = (df_enhanced['PreRwaist'] >= 50) & (df_enhanced['PreRwaist'] <= 200)  # cm
        valid_hip = (df_enhanced['PreRhip'] >= 60) & (df_enhanced['PreRhip'] <= 200)        # cm
        
        df_enhanced['WHR'] = np.where(
            valid_waist & valid_hip & (df_enhanced['PreRhip'] > 0),
            df_enhanced['PreRwaist'] / df_enhanced['PreRhip'],
            np.nan
        )
        
        # WHR risk categories (gender-specific WHO criteria)
        def categorize_whr(whr, gender):
            """Categorize WHR risk based on gender-specific WHO criteria"""
            if pd.isna(whr):
                return 'Unknown'
            
            # Gender: 1=Male, 0=Female (assuming this encoding)
            if gender == 1:  # Male
                if whr < 0.90:
                    return 'Low_Risk'
                elif whr <= 0.99:
                    return 'Moderate_Risk'
                else:
                    return 'High_Risk'
            else:  # Female  
                if whr < 0.80:
                    return 'Low_Risk'
                elif whr <= 0.84:
                    return 'Moderate_Risk'
                else:
                    return 'High_Risk'
        
        if 'PreRgender' in df.columns:
            df_enhanced['WHR_Risk'] = df_enhanced.apply(
                lambda row: categorize_whr(row['WHR'], row['PreRgender']), axis=1
            )
            features_created.extend(['WHR', 'WHR_Risk'])
            print(f"   ✅ WHR: Valid calculations for {df_enhanced['WHR'].notna().sum():,} samples")
            whr_risk_counts = df_enhanced['WHR_Risk'].value_counts()
            print(f"   📊 WHR Risk: {whr_risk_counts.to_dict()}")
        else:
            features_created.append('WHR')
            print("   ⚠️  Gender column not found - creating WHR only")
    else:
        print("   ⚠️  Waist/hip columns not found - skipping WHR calculation")
    
    # 4. Metabolic Syndrome Indicators (ATP III criteria)
    metabolic_components = []
    if 'PreBLFBS' in df.columns:
        print("🧪 Creating metabolic syndrome indicators...")
        
        # Glucose criteria (≥100 mg/dL or ≥5.6 mmol/L)
        df_enhanced['High_FBS'] = (df_enhanced['PreBLFBS'] >= 100).astype(int)
        metabolic_components.append('High_FBS')
        
    if 'PreBLTRIGLYCERIDES' in df.columns:
        # Triglycerides criteria (≥150 mg/dL)
        df_enhanced['High_Triglycerides'] = (df_enhanced['PreBLTRIGLYCERIDES'] >= 150).astype(int)
        metabolic_components.append('High_Triglycerides')
        
    # Note: Using total cholesterol as HDL proxy (limitation - should ideally use HDL-C)
    if 'PreBLCHOLESTEROL' in df.columns:
        # Assuming low total cholesterol might indicate metabolic issues
        df_enhanced['Abnormal_Cholesterol'] = (
            (df_enhanced['PreBLCHOLESTEROL'] < 150) | 
            (df_enhanced['PreBLCHOLESTEROL'] > 240)
        ).astype(int)
        metabolic_components.append('Abnormal_Cholesterol')
        
    # Blood pressure component
    if 'BP_Category' in df_enhanced.columns:
        df_enhanced['High_BP'] = df_enhanced['BP_Category'].isin(['Stage_1_HTN', 'Stage_2_HTN']).astype(int)
        metabolic_components.append('High_BP')
        
    # Central obesity component
    if 'WHR_Risk' in df_enhanced.columns:
        df_enhanced['High_WHR'] = df_enhanced['WHR_Risk'].isin(['High_Risk']).astype(int)
        metabolic_components.append('High_WHR')
    elif 'BMI' in df_enhanced.columns:
        df_enhanced['High_BMI'] = (df_enhanced['BMI'] >= 25).astype(int)  # Asian cutoff
        metabolic_components.append('High_BMI')
        
    # Calculate overall metabolic syndrome score (0-5 scale)
    if metabolic_components:
        df_enhanced['Metabolic_Syndrome_Score'] = df_enhanced[metabolic_components].sum(axis=1)
        metabolic_components.append('Metabolic_Syndrome_Score')
        features_created.extend(metabolic_components)
        
        ms_score_dist = df_enhanced['Metabolic_Syndrome_Score'].value_counts().sort_index()
        print(f"   ✅ Metabolic Syndrome Score distribution: {ms_score_dist.to_dict()}")
    else:
        print("   ⚠️  Insufficient lab data for metabolic syndrome scoring")
    
    # 5. Lifestyle Risk Score
    print("🏃 Creating lifestyle risk assessment...")
    lifestyle_risk_features = []
    
    # Smoking risk (if available)
    smoking_cols = [col for col in df.columns if 'smoking' in col.lower() or 'tobacco' in col.lower()]
    if smoking_cols:
        # Use first available smoking column
        smoking_col = smoking_cols[0]
        if df_enhanced[smoking_col].dtype == 'object':
            # Map categorical responses to risk scores
            smoking_map = {'Yes': 2, 'Current': 2, 'Former': 1, 'No': 0, 'Never': 0}
            df_enhanced['Smoking_Risk'] = df_enhanced[smoking_col].map(smoking_map).fillna(1)
        else:
            df_enhanced['Smoking_Risk'] = df_enhanced[smoking_col].fillna(0)
        lifestyle_risk_features.append('Smoking_Risk')
    
    # Alcohol risk (if available)
    alcohol_cols = [col for col in df.columns if 'alcohol' in col.lower()]
    if alcohol_cols:
        alcohol_col = alcohol_cols[0]
        if df_enhanced[alcohol_col].dtype == 'object':
            # Map categorical responses to risk scores
            alcohol_map = {'Heavy': 2, 'Moderate': 1, 'Light': 0, 'No': 0, 'Never': 0}
            df_enhanced['Alcohol_Risk'] = df_enhanced[alcohol_col].map(alcohol_map).fillna(1)
        else:
            df_enhanced['Alcohol_Risk'] = df_enhanced[alcohol_col].fillna(0)
        lifestyle_risk_features.append('Alcohol_Risk')
    
    # Diet risk factors
    diet_risk_cols = ['PreRskipbreakfast', 'PreRlessfiber', 'PreRlessfruit', 'PreRlessvegetable']
    available_diet_cols = [col for col in diet_risk_cols if col in df.columns]
    
    if available_diet_cols:
        for col in available_diet_cols:
            risk_col = f"{col}_risk"
            # Map dietary habits to risk scores
            df_enhanced[risk_col] = df_enhanced[col].map({
                'Usually/Often': 2,
                'Usually': 2,
                'Often': 2, 
                'Sometimes': 1, 
                'Rarely': 0,
                'Never': 0,
                'Rarely/ Never': 0
            }).fillna(1)  # Default to moderate risk for missing values
            lifestyle_risk_features.append(risk_col)
    
    # Physical activity protective factor
    activity_cols = [col for col in df.columns if 'activity' in col.lower() or 'exercise' in col.lower()]
    if activity_cols:
        activity_col = activity_cols[0]
        if 'Optimal' in str(df_enhanced[activity_col].unique()):
            df_enhanced['Low_Physical_Activity'] = (df_enhanced[activity_col] != 'Optimal').astype(int)
        else:
            # Assume numeric or binary encoding
            df_enhanced['Low_Physical_Activity'] = (df_enhanced[activity_col] == 0).astype(int)
        lifestyle_risk_features.append('Low_Physical_Activity')
    
    # Calculate overall lifestyle risk score
    if lifestyle_risk_features:
        df_enhanced['Lifestyle_Risk_Score'] = df_enhanced[lifestyle_risk_features].sum(axis=1)
        lifestyle_risk_features.append('Lifestyle_Risk_Score')
        features_created.extend(lifestyle_risk_features)
        
        lifestyle_dist = df_enhanced['Lifestyle_Risk_Score'].value_counts().sort_index()
        print(f"   ✅ Lifestyle Risk Score from {len(lifestyle_risk_features)-1} components: {lifestyle_dist.to_dict()}")
    else:
        print("   ⚠️  Limited lifestyle data available")
    
    # 6. Age-Gender Interaction (important for diabetes risk)
    if 'PreBLAge' in df.columns and 'PreRgender' in df.columns:
        print("👥 Creating demographic interaction features...")
        df_enhanced['Age_Gender_Interaction'] = df_enhanced['PreBLAge'] * df_enhanced['PreRgender']
        
        # Age risk categories for diabetes
        df_enhanced['Age_Risk_Category'] = pd.cut(
            df_enhanced['PreBLAge'],
            bins=[0, 45, 65, 100],
            labels=['Low_Risk', 'Moderate_Risk', 'High_Risk'],
            include_lowest=True
        )
        
        features_created.extend(['Age_Gender_Interaction', 'Age_Risk_Category'])
        age_dist = df_enhanced['Age_Risk_Category'].value_counts()
        print(f"   ✅ Age Risk Categories: {age_dist.to_dict()}")
    else:
        print("   ⚠️  Age or gender columns not found")
    
    # 7. Family History Score (genetic predisposition)
    family_history_cols = ['PreRdiafather', 'PreRdiamother', 'PreRdiabrother', 'PreRdiasister']
    available_fh_cols = [col for col in family_history_cols if col in df.columns]
    
    if available_fh_cols:
        print("👨‍👩‍👧‍👦 Creating family history features...")
        
        # Convert to binary if needed and sum
        for col in available_fh_cols:
            if df_enhanced[col].dtype == 'object':
                df_enhanced[col] = (df_enhanced[col] == 'Yes').astype(int)
        
        df_enhanced['Family_History_Score'] = df_enhanced[available_fh_cols].sum(axis=1)
        
        # Family history risk categories
        df_enhanced['Family_History_Risk'] = pd.cut(
            df_enhanced['Family_History_Score'],
            bins=[-1, 0, 1, 4],
            labels=['No_Risk', 'Low_Risk', 'High_Risk'],
            include_lowest=True
        )
        
        features_created.extend(['Family_History_Score', 'Family_History_Risk'])
        fh_dist = df_enhanced['Family_History_Score'].value_counts().sort_index()
        print(f"   ✅ Family History Score from {len(available_fh_cols)} relatives: {fh_dist.to_dict()}")
    else:
        print("   ⚠️  Family history columns not found")
    
    # 8. Clinical Lab Risk Stratification
    if 'PreBLHBA1C' in df.columns:
        print("🧪 Creating baseline HbA1c risk categories...")
        df_enhanced['Baseline_HbA1c_Risk'] = pd.cut(
            df_enhanced['PreBLHBA1C'],
            bins=[0, 5.7, 6.5, 7.0, 20],
            labels=['Normal', 'Prediabetes', 'Diabetes_Controlled', 'Diabetes_Uncontrolled'],
            include_lowest=True
        )
        features_created.append('Baseline_HbA1c_Risk')
        
        hba1c_dist = df_enhanced['Baseline_HbA1c_Risk'].value_counts()
        print(f"   ✅ Baseline HbA1c Risk: {hba1c_dist.to_dict()}")
    
    print(f"\n📊 FEATURE ENGINEERING SUMMARY:")
    print(f"   Original features: {len(df.columns)}")
    print(f"   New features created: {len(features_created)}")
    print(f"   Total features: {len(df_enhanced.columns)}")
    print(f"   Memory usage: {df_enhanced.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
    
    if features_created:
        print(f"\n📋 New Features Created:")
        for i, feature in enumerate(features_created, 1):
            print(f"   {i:2d}. {feature}")
    
    return df_enhanced, features_created


def prepare_features_for_automl(df: pd.DataFrame, config: Dict[str, object]) -> Tuple[pd.DataFrame, pd.Series, Dict[str, object]]:
    """
    Prepare features for H2O AutoML with proper encoding and validation.
    
    This function:
    - Separates target variable and removes identifier columns
    - Analyzes feature correlations with target
    - Identifies multicollinearity issues
    - Provides feature type summaries for AutoML
    
    Args:
        df: Enhanced dataset with engineered features.
        config: Notebook configuration dictionary.
    
    Returns:
        Tuple containing (feature_df, target_series, feature_info).
    """
    
    print("\n🎯 PREPARING FEATURES FOR AUTOML")
    print("=" * 40)
    
    target_col = config['target_column']
    df_ml = df.copy()
    
    if target_col not in df_ml.columns:
        available_cols = ', '.join(df_ml.columns[:10].tolist())
        raise KeyError(f"Target column '{target_col}' not found. Available columns include: {available_cols}...")
    
    # Separate target variable
    target = df_ml[target_col].copy()
    feature_df = df_ml.drop(columns=[target_col])
    
    print(f"🎯 Target Variable Analysis:")
    print(f"   Column: {target_col}")
    print(f"   Non-null values: {target.notna().sum():,} / {len(target):,}")
    print(f"   Range: {target.min():.2f} - {target.max():.2f}")
    print(f"   Mean ± SD: {target.mean():.2f} ± {target.std():.2f}")
    
    # Remove identifier columns that shouldn't be used for prediction
    identifier_cols = config.get('modeling', {}).get('identifier_columns', [])
    identifier_cols_found = [col for col in identifier_cols if col in feature_df.columns]
    if identifier_cols_found:
        feature_df = feature_df.drop(columns=identifier_cols_found)
        print(f"🗂️  Removed identifier columns: {identifier_cols_found}")
    
    # Analyze feature types
    categorical_cols = feature_df.select_dtypes(include=['object', 'category']).columns.tolist()
    numerical_cols = feature_df.select_dtypes(include=[np.number]).columns.tolist()
    
    print(f"\n📋 Feature Type Summary:")
    print(f"   Numerical features: {len(numerical_cols)}")
    print(f"   Categorical features: {len(categorical_cols)}")
    print(f"   Total features for modeling: {len(feature_df.columns)}")
    
    # Handle categorical variables for correlation analysis
    feature_df_encoded = feature_df.copy()
    
    categorical_encoding_info = {}
    for col in categorical_cols:
        if feature_df_encoded[col].dtype == 'object':
            # Check unique values before encoding
            unique_vals = feature_df_encoded[col].nunique()
            categorical_encoding_info[col] = unique_vals
            
            if unique_vals > 50:
                print(f"   ⚠️  High cardinality categorical: {col} ({unique_vals} categories)")
            
            # FIX: Convert all values to strings first, then use LabelEncoder
            # This handles mixed float/string columns properly
            feature_df_encoded[col] = feature_df_encoded[col].astype(str).fillna('Unknown')
            
            # Use label encoding for correlation analysis (H2O handles categoricals internally)
            le = LabelEncoder()
            feature_df_encoded[col] = le.fit_transform(feature_df_encoded[col])

    # Feature correlation analysis with target
    print(f"\n🔍 Feature-Target Correlation Analysis:")
    
    try:
        correlations = feature_df_encoded.corrwith(target).abs().sort_values(ascending=False)
        
        # Remove NaN correlations (constant features, etc.)
        correlations = correlations.dropna()
        
        print(f"\n🎯 TOP 15 FEATURES CORRELATED WITH {target_col}:")
        for i, (feature, corr) in enumerate(correlations.head(15).items(), 1):
            correlation_strength = "Strong" if corr > 0.5 else "Moderate" if corr > 0.3 else "Weak"
            print(f"   {i:2d}. {feature:<35} {corr:.3f} ({correlation_strength})")
            
    except Exception as e:
        print(f"   ⚠️  Error computing correlations: {str(e)}")
        correlations = pd.Series(dtype=float)
    
    # Identify highly correlated feature pairs (multicollinearity check)
    print(f"\n🔍 Multicollinearity Analysis:")
    
    try:
        # Sample data if too large for correlation matrix
        if len(feature_df_encoded) > 5000:
            sample_df = feature_df_encoded.sample(n=5000, random_state=42)
        else:
            sample_df = feature_df_encoded
            
        corr_matrix = sample_df.corr()
        high_corr_pairs = []
        
        for i in range(len(corr_matrix.columns)):
            for j in range(i+1, len(corr_matrix.columns)):
                corr_val = abs(corr_matrix.iloc[i, j])
                if corr_val > 0.8:  # High correlation threshold
                    high_corr_pairs.append((
                        corr_matrix.columns[i], 
                        corr_matrix.columns[j], 
                        corr_val
                    ))
        
        if high_corr_pairs:
            print(f"   ⚠️  High Correlation Pairs (>0.8) - Consider feature selection:")
            for feat1, feat2, corr_val in sorted(high_corr_pairs, key=lambda x: x[2], reverse=True)[:10]:
                print(f"      {feat1} ↔ {feat2}: {corr_val:.3f}")
                
            if len(high_corr_pairs) > 10:
                print(f"      ... +{len(high_corr_pairs) - 10} more pairs")
        else:
            print(f"   ✅ No high correlation pairs detected (threshold: 0.8)")
            
    except Exception as e:
        print(f"   ⚠️  Error computing feature correlations: {str(e)}")
        high_corr_pairs = []
    
    # Memory usage check
    memory_mb = feature_df.memory_usage(deep=True).sum() / 1024**2
    print(f"\n💾 Memory Usage: {memory_mb:.1f} MB")
    
    if memory_mb > 500:
        print(f"   ⚠️  Large dataset - consider feature selection for faster training")
    
    # Feature information summary
    feature_info = {
        'numerical_features': numerical_cols,
        'categorical_features': categorical_cols,
        'categorical_encoding_info': categorical_encoding_info,
        'total_features': len(feature_df.columns),
        'target_correlations': correlations.to_dict() if len(correlations) > 0 else {},
        'high_correlation_pairs': high_corr_pairs,
        'memory_usage_mb': memory_mb
    }
    
    print(f"\n✅ Feature preparation completed successfully")
    print(f"   📊 Ready for AutoML: {len(feature_df.columns)} features, {len(target)} samples")
    print(f"   🎯 Target completeness: {target.notna().mean()*100:.1f}%")
    
    return feature_df, target, feature_info


# Execute feature engineering pipeline
print("🚀 Starting Advanced Clinical Feature Engineering Pipeline...")

try:
    # Create clinical features with comprehensive error handling
    df_enhanced, new_features = create_clinical_features(df_main)
    
    # Prepare features for AutoML
    X, y, feature_info = prepare_features_for_automl(df_enhanced, NOTEBOOK_CONFIG)
    
    print(f"\n🎉 FEATURE ENGINEERING PIPELINE COMPLETED!")
    print(f"📊 Final Dataset Specifications:")
    print(f"   • Samples: {X.shape[0]:,}")
    print(f"   • Features: {X.shape[1]:,}")
    print(f"   • New clinical features: {len(new_features)}")
    print(f"   • Target variable: {NOTEBOOK_CONFIG['target_column']}")
    print(f"   • Memory footprint: {feature_info['memory_usage_mb']:.1f} MB")
    print(f"\n✅ Dataset is ready for H2O AutoML training!")
    
except Exception as exc:
    print(f"❌ CRITICAL ERROR in feature engineering: {exc}")
    print("🔍 Please check your dataset structure and column names")
    raise

In [ ]:
# =============================================================================
# H2O AUTOML SETUP AND CONFIGURATION
# =============================================================================

def initialize_h2o_cluster():
    """
    Initialize H2O cluster with optimized settings for Google Colab
    """
    
    print("🚀 INITIALIZING H2O CLUSTER")
    print("=" * 40)
    
    try:
        # Initialize H2O with Colab-optimized settings
        h2o.init(
            nthreads=-1,          # Use all available cores
            max_mem_size="6G",    # Colab memory limit
            port=54321,           # Default H2O port
            name="diabetes-automl"
        )
        
        # Display cluster info (compatible with H2O 3.46+)
        try:
            cluster_status = h2o.cluster_status()
            print(f"✅ H2O Cluster initialized successfully")
            print(f"   Version: {h2o.__version__}")
            print(f"   Nodes: 1")
            print(f"   Memory: 6G (configured)")
            print(f"   Cores: {cluster_status.cores if hasattr(cluster_status, 'cores') else 'Available'}")
        except:
            print(f"✅ H2O Cluster initialized successfully")
            print(f"   Version: {h2o.__version__}")
            print(f"   Status: Running")
        
        return True
        
    except Exception as e:
        print(f"❌ Error initializing H2O cluster: {str(e)}")
        print("⚠️  Attempting to connect to existing cluster...")
        
        try:
            h2o.connect()
            print("✅ Connected to existing H2O cluster")
            return True
        except:
            print("❌ Failed to connect to H2O cluster")
            return False


def prepare_h2o_data(X: pd.DataFrame, y: pd.Series, config: Dict[str, object]) -> Tuple[h2o.H2OFrame, h2o.H2OFrame, h2o.H2OFrame, List[str], str]:
    """
    Prepare data for H2O AutoML with train/validation/test splits.
    
    Args:
        X: Feature matrix.
        y: Target variable series.
        config: Notebook configuration dictionary.
    
    Returns:
        Tuple of (h2o_train, h2o_valid, h2o_test, feature_names, target_name).
    """
    
    print("\n📊 PREPARING DATA FOR H2O AUTOML")
    print("=" * 40)
    
    target_name = config['target_column']
    split_cfg = config.get('train_test_split', {})
    test_size = split_cfg.get('test_size', 0.2)
    validation_fraction = split_cfg.get('validation_fraction', 0.2)
    random_state = split_cfg.get('random_state', 42)
    stratify_bins = split_cfg.get('stratify_bins', 5)
    
    # Combine features and target
    df_combined = X.copy()
    df_combined[target_name] = y
    
    print(f"📋 Data Summary:")
    print(f"   Total samples: {len(df_combined):,}")
    print(f"   Features: {len(X.columns)}")
    print(f"   Target: {target_name}")
    
    # Prepare stratification labels if feasible
    stratify_labels = None
    try:
        stratify_labels = pd.qcut(y, q=stratify_bins, duplicates='drop')
    except ValueError:
        print("⚠️  Stratified split skipped (insufficient unique values). Using random split instead.")
    
    # Split data into train and test
    train_data, test_data = train_test_split(
        df_combined,
        test_size=test_size,
        random_state=random_state,
        stratify=stratify_labels
    )
    
    # Further split train into train and validation
    stratify_train = None
    if stratify_labels is not None:
        stratify_train = pd.qcut(train_data[target_name], q=stratify_bins, duplicates='drop')
    
    train_data, valid_data = train_test_split(
        train_data,
        test_size=validation_fraction,
        random_state=random_state,
        stratify=stratify_train
    )
    
    print(f"\n📊 Data Splits:")
    print(f"   Training: {len(train_data):,} samples ({len(train_data)/len(df_combined)*100:.1f}%)")
    print(f"   Validation: {len(valid_data):,} samples ({len(valid_data)/len(df_combined)*100:.1f}%)")
    print(f"   Test: {len(test_data):,} samples ({len(test_data)/len(df_combined)*100:.1f}%)")
    
    # Convert to H2O frames
    print("\n🔄 Converting to H2O frames...")
    
    h2o_train = h2o.H2OFrame(train_data)
    h2o_valid = h2o.H2OFrame(valid_data)
    h2o_test = h2o.H2OFrame(test_data)
    
    feature_names = [col for col in h2o_train.columns if col != target_name]
    
    # Convert categorical features to factors (only string/object columns)
    categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
    
    for col in categorical_features:
        if col in h2o_train.columns:
            # Check if column type is suitable for factor conversion
            col_type = h2o_train[col].types[col]
            if col_type in ['string', 'enum']:
                h2o_train[col] = h2o_train[col].asfactor()
                h2o_valid[col] = h2o_valid[col].asfactor()
                h2o_test[col] = h2o_test[col].asfactor()
            else:
                print(f"   ⚠️  Skipping {col} (type: {col_type}) - not suitable for factor conversion")
    
    print(f"✅ H2O data preparation completed")
    print(f"   Categorical features: {len(categorical_features)}")
    print(f"   Numerical features: {len(feature_names) - len(categorical_features)}")
    
    return h2o_train, h2o_valid, h2o_test, feature_names, target_name


def configure_automl(config: Dict[str, object]) -> Dict[str, object]:
    """
    Configure H2O AutoML with medical-specific parameters.
    
    Args:
        config: Notebook configuration dictionary.
    
    Returns:
        AutoML configuration parameters.
    """
    
    print("\n⚙️ CONFIGURING H2O AUTOML")
    print("=" * 40)
    
    automl_cfg = {
        'max_models': config.get('automl', {}).get('max_models', 20),
        'max_runtime_secs': config.get('automl', {}).get('max_runtime_secs', 3600),
        'seed': config.get('automl', {}).get('seed', 42),
        'nfolds': config.get('automl', {}).get('nfolds', 5),
        'sort_metric': config.get('automl', {}).get('sort_metric', 'RMSE'),
        'stopping_metric': config.get('automl', {}).get('sort_metric', 'RMSE'),
        'stopping_tolerance': config.get('automl', {}).get('stopping_tolerance', 0.001),
        'stopping_rounds': config.get('automl', {}).get('stopping_rounds', 3),
        'project_name': config.get('automl', {}).get('project_name', 'diabetes_hba1c_automl'),
    }
    
    print(f"📋 AutoML Configuration:")
    for key, value in automl_cfg.items():
        print(f"   {key}: {value}")
    
    print(f"\n🧠 Algorithm Coverage:")
    print(f"   • Gradient Boosting Machine (GBM)")
    print(f"   • Extreme Gradient Boosting (XGBoost)")
    print(f"   • Random Forest (DRF)")
    print(f"   • Generalized Linear Model (GLM)")
    print(f"   • Deep Learning Neural Networks")
    print(f"   • Stacked Ensemble Models")
    
    return automl_cfg


def run_automl(h2o_train, h2o_valid, h2o_test, feature_names, target_name, config):
    """
    Execute H2O AutoML training with comprehensive monitoring
    
    Args:
        h2o_train (H2OFrame): Training data
        h2o_valid (H2OFrame): Validation data
        h2o_test (H2OFrame): Test data
        feature_names (list): List of feature column names
        target_name (str): Target variable name
        config (dict): AutoML configuration
        
    Returns:
        H2OAutoML: Trained AutoML object
    """
    
    print("\n🤖 EXECUTING H2O AUTOML TRAINING")
    print("=" * 50)
    
    start_time = time.time()
    
    try:
        # Initialize AutoML
        automl = H2OAutoML(
            max_models=config['max_models'],
            max_runtime_secs=config['max_runtime_secs'],
            seed=config['seed'],
            nfolds=config['nfolds'],
            sort_metric=config['sort_metric'],
            stopping_metric=config['stopping_metric'],
            stopping_tolerance=config['stopping_tolerance'],
            stopping_rounds=config['stopping_rounds'],
            project_name=config['project_name']
        )
        
        print(f"🚀 Starting AutoML training...")
        print(f"   Max models: {config['max_models']}")
        print(f"   Max runtime: {config['max_runtime_secs']//60} minutes")
        print(f"   Cross-validation folds: {config['nfolds']}")
        
        # Train AutoML
        automl.train(
            x=feature_names,
            y=target_name,
            training_frame=h2o_train,
            validation_frame=h2o_valid,
            leaderboard_frame=h2o_test
        )
        
        training_time = time.time() - start_time
        
        print(f"\n✅ AutoML training completed successfully!")
        print(f"⏱️  Training time: {training_time/60:.1f} minutes")
        print(f"📊 Models trained: {len(automl.leaderboard)}")
        
        # Display leaderboard summary
        leaderboard = automl.leaderboard.as_data_frame()
        print(f"\n🏆 TOP 5 MODELS PERFORMANCE:")
        print("=" * 80)
        
        display_columns = ['model_id', 'rmse', 'mae', 'mean_residual_deviance']
        available_columns = [col for col in display_columns if col in leaderboard.columns]
        
        for i, (_, row) in enumerate(leaderboard[available_columns].head(5).iterrows(), 1):
            model_id = row['model_id']
            rmse = row.get('rmse', 'N/A')
            mae = row.get('mae', 'N/A')
            
            print(f"{i}. {model_id[:50]}...")
            if rmse != 'N/A':
                print(f"   RMSE: {rmse:.4f} | MAE: {mae:.4f}")
        
        # Get best model
        best_model = automl.leader
        print(f"\n🥇 BEST MODEL: {best_model.model_id}")
        
        return automl
        
    except Exception as e:
        print(f"❌ AutoML training failed: {str(e)}")
        raise


def evaluate_model_performance(automl, h2o_test, target_name, config):
    """
    Comprehensive model evaluation with clinical interpretability
    
    Args:
        automl: Trained H2O AutoML object
        h2o_test: H2O test dataset
        target_name: Target column name
        config: Configuration dictionary
        
    Returns:
        dict: Performance metrics
    """
    
    print("\n? MODEL PERFORMANCE EVALUATION")
    print("=" * 50)
    
    best_model = automl.leader
    
    # Generate predictions
    predictions = best_model.predict(h2o_test)
    
    # Convert to pandas for analysis
    y_true = h2o_test[target_name].as_data_frame().iloc[:, 0]
    y_pred = predictions.as_data_frame().iloc[:, 0]
    
    # Calculate comprehensive metrics
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    
    metrics = {
        'rmse': rmse,
        'mae': mae,
        'r2_score': r2,
        'mape': mape
    }
    
    print(f"🏆 PERFORMANCE METRICS:")
    print(f"   RMSE: {rmse:.4f} %")
    print(f"   MAE: {mae:.4f} %")
    print(f"   R²: {r2:.4f}")
    print(f"   MAPE: {mape:.2f}%")
    
    # Clinical accuracy thresholds
    tolerance_primary = 0.5  # ±0.5% HbA1c tolerance
    tolerance_secondary = 1.0  # ±1.0% HbA1c tolerance
    
    within_primary = np.abs(y_true - y_pred) <= tolerance_primary
    within_secondary = np.abs(y_true - y_pred) <= tolerance_secondary
    
    primary_accuracy = within_primary.mean() * 100
    secondary_accuracy = within_secondary.mean() * 100
    
    print(f"\n🏥 CLINICAL ACCURACY:")
    print(f"   Within ±0.5%: {primary_accuracy:.1f}% of predictions")
    print(f"   Within ±1.0%: {secondary_accuracy:.1f}% of predictions")
    
    metrics.update({
        'clinical_accuracy_0.5': primary_accuracy,
        'clinical_accuracy_1.0': secondary_accuracy
    })
    
    # Visualization
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Model Performance Analysis', fontsize=16, fontweight='bold')
    
    # Actual vs Predicted
    axes[0, 0].scatter(y_true, y_pred, alpha=0.6, color='steelblue')
    min_val, max_val = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
    axes[0, 0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2)
    axes[0, 0].set_xlabel('Actual HbA1c (%)')
    axes[0, 0].set_ylabel('Predicted HbA1c (%)')
    axes[0, 0].set_title(f'Actual vs Predicted (R² = {r2:.3f})')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Residuals plot
    residuals = y_pred - y_true
    axes[0, 1].scatter(y_pred, residuals, alpha=0.6, color='orange')
    axes[0, 1].axhline(y=0, color='red', linestyle='--')
    axes[0, 1].set_xlabel('Predicted HbA1c (%)')
    axes[0, 1].set_ylabel('Residuals')
    axes[0, 1].set_title(f'Residuals Plot (RMSE = {rmse:.3f})')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Distribution of residuals
    axes[1, 0].hist(residuals, bins=30, alpha=0.7, color='lightcoral', edgecolor='black')
    axes[1, 0].axvline(residuals.mean(), color='red', linestyle='--', label=f'Mean: {residuals.mean():.3f}')
    axes[1, 0].set_xlabel('Residuals')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Distribution of Residuals')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Clinical accuracy visualization
    error_abs = np.abs(residuals)
    within_primary = (error_abs <= tolerance_primary).sum()
    within_secondary = (error_abs <= tolerance_secondary).sum()
    outside_secondary = len(error_abs) - within_secondary
    
    categories = ['Within ±0.5%', 'Within ±1.0%', 'Outside ±1.0%']
    counts = [within_primary, within_secondary - within_primary, outside_secondary]
    colors = ['lightgreen', 'gold', 'lightcoral']
    
    axes[1, 1].pie(counts, labels=categories, colors=colors, autopct='%1.1f%%', startangle=90)
    axes[1, 1].set_title('Clinical Accuracy Distribution')
    
    plt.tight_layout()
    plt.show()
    
    return metrics


# Execute H2O AutoML Pipeline
print("🚀 Starting H2O AutoML Pipeline for HbA1c Prediction...")

# Initialize H2O cluster
if initialize_h2o_cluster():
    # Prepare data for H2O
    h2o_train, h2o_valid, h2o_test, feature_names, target_name = prepare_h2o_data(X, y, NOTEBOOK_CONFIG)

    # Configure AutoML
    automl_config = configure_automl(NOTEBOOK_CONFIG)

    # Run AutoML
    automl = run_automl(
        h2o_train, h2o_valid, h2o_test,
        feature_names, target_name,
        automl_config
    )

    # Evaluate model performance
    performance_metrics = evaluate_model_performance(automl, h2o_test, target_name, NOTEBOOK_CONFIG)

    print(f"\n🎉 AutoML pipeline completed successfully!")
    print("📈 Ready for feature importance analysis and final reporting")
else:
    print("❌ Failed to initialize H2O cluster. Please check your environment.")

In [ ]:
# =============================================================================
# FEATURE IMPORTANCE & CLINICAL INTERPRETABILITY ANALYSIS
# =============================================================================

def analyze_feature_importance(automl, feature_names):
    """
    Comprehensive feature importance analysis with clinical interpretation

    Args:
        automl: Trained H2O AutoML object
        feature_names: List of feature names

    Returns:
        dict: Feature importance insights
    """

    print("🔬 FEATURE IMPORTANCE ANALYSIS")
    print("=" * 50)

    best_model = automl.leader

    try:
        var_importance = best_model.varimp(use_pandas=True)

        print(f"🏆 TOP 20 MOST IMPORTANT FEATURES:")
        print("=" * 70)

        clinical_categories = {
            'Clinical Labs': ['FBS', 'PPBS', 'HBA1C', 'CHOLESTEROL', 'TRIGLYCERIDES', 'LDL'],
            'Anthropometric': ['BMI', 'WEIGHT', 'HEIGHT', 'WAIST', 'HIP', 'WHR'],
            'Vital Signs': ['SYSTOLIC', 'DIASTOLIC', 'BP_CATEGORY'],
            'Demographics': ['AGE', 'GENDER', 'AREA', 'EDUCATION', 'OCCUPATION'],
            'Lifestyle': ['SMOKING', 'ALCOHOL', 'PHYSICAL_ACTIVITY', 'SLEEP'],
            'Family History': ['DIAFATHER', 'DIAMOTHER', 'DIABROTHER', 'DIASISTER'],
            'Metabolic Risk': ['METABOLIC_SYNDROME', 'HIGH_FBS', 'HIGH_BP', 'HIGH_WHR'],
            'Derived Features': ['LIFESTYLE_RISK', 'FAMILY_HISTORY_SCORE', 'AGE_GENDER']
        }

        def categorize_feature(feature_name):
            feature_upper = feature_name.upper()
            for category, keywords in clinical_categories.items():
                if any(keyword in feature_upper for keyword in keywords):
                    return category
            return 'Other'

        category_counts = {}
        for i, (_, row) in enumerate(var_importance.head(20).iterrows(), 1):
            feature = row['variable']
            importance = row['relative_importance']
            scaled_importance = row['scaled_importance']

            category = categorize_feature(feature)
            category_counts[category] = category_counts.get(category, 0) + 1

            print(f"{i:2d}. {feature:<35} {importance:8.4f} ({scaled_importance:6.1f}%) [{category}]")

        print(f"\n📊 FEATURE IMPORTANCE BY CLINICAL CATEGORY:")
        print("=" * 60)

        category_importance = {}
        for category in clinical_categories.keys():
            category_features = [row['variable'] for _, row in var_importance.iterrows()
                                 if categorize_feature(row['variable']) == category]

            if category_features:
                total_importance = sum([row['relative_importance'] for _, row in var_importance.iterrows()
                                        if row['variable'] in category_features])
                category_importance[category] = total_importance

                avg_importance = total_importance / len(category_features)

                print(f"{category:<20} | Features: {len(category_features):2d} | "
                      f"Total: {total_importance:8.4f} | Avg: {avg_importance:.4f}")

        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('Feature Importance Analysis', fontsize=16, fontweight='bold')

        top_features = var_importance.head(15)
        bars = axes[0, 0].barh(range(len(top_features)), top_features['scaled_importance'])
        axes[0, 0].set_yticks(range(len(top_features)))
        axes[0, 0].set_yticklabels([f"{feat[:25]}..." if len(feat) > 25 else feat
                                   for feat in top_features['variable']], fontsize=9)
        axes[0, 0].set_xlabel('Scaled Importance (%)')
        axes[0, 0].set_title('Top 15 Features')
        axes[0, 0].invert_yaxis()

        colors = ['red', 'blue', 'green', 'orange', 'purple', 'brown', 'pink', 'gray']
        category_colors = {cat: colors[i % len(colors)] for i, cat in enumerate(clinical_categories.keys())}

        for i, (bar, feat) in enumerate(zip(bars, top_features['variable'])):
            category = categorize_feature(feat)
            bar.set_color(category_colors.get(category, 'gray'))

        axes[0, 0].grid(True, alpha=0.3)

        if category_importance:
            labels = list(category_importance.keys())
            sizes = list(category_importance.values())
            colors_pie = [category_colors.get(cat, 'gray') for cat in labels]

            axes[0, 1].pie(sizes, labels=labels, colors=colors_pie,
                           autopct='%1.1f%%', startangle=90)
            axes[0, 1].set_title('Importance by Clinical Category')

        cumulative_importance = np.cumsum(var_importance['scaled_importance'])
        axes[1, 0].plot(range(1, len(cumulative_importance) + 1), cumulative_importance,
                        marker='o', linewidth=2, markersize=4)
        axes[1, 0].axhline(y=80, color='red', linestyle='--', alpha=0.7, label='80% threshold')
        axes[1, 0].axhline(y=90, color='orange', linestyle='--', alpha=0.7, label='90% threshold')
        axes[1, 0].set_xlabel('Number of Features')
        axes[1, 0].set_ylabel('Cumulative Importance (%)')
        axes[1, 0].set_title('Cumulative Feature Importance')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)

        category_counts_plot = {k: v for k, v in category_counts.items() if v > 0}
        if category_counts_plot:
            axes[1, 1].bar(category_counts_plot.keys(), category_counts_plot.values(),
                          color=[category_colors.get(cat, 'gray') for cat in category_counts_plot.keys()])
            axes[1, 1].set_ylabel('Number of Top Features')
            axes[1, 1].set_title('Top Features Count by Category')
            axes[1, 1].tick_params(axis='x', rotation=45)

        plt.tight_layout()
        plt.show()

        return {
            'variable_importance': var_importance,
            'category_importance': category_importance,
            'category_counts': category_counts
        }

    except Exception as e:
        print(f"❌ Error getting variable importance: {str(e)}")
        print("⚠️  Variable importance may not be available for this model type")
        return None


def generate_clinical_insights(importance_results, performance_metrics, config):
    """
    Generate actionable clinical insights from model results

    Args:
        importance_results: Feature importance analysis results
        performance_metrics: Model performance metrics
        config: Notebook configuration dictionary
    """

    print("\n🏥 CLINICAL INSIGHTS & RECOMMENDATIONS")
    print("=" * 60)

    evaluation_cfg = config.get("evaluation_thresholds", {})
    clinical_cfg = config.get("clinical_thresholds", {})

    rmse_target = evaluation_cfg.get("excellent_rmse", 0.5)
    clinical_target = evaluation_cfg.get("clinical_accuracy_target", 80.0)
    clinical_watch = evaluation_cfg.get("clinical_accuracy_watch", 70.0)
    tolerance_primary = clinical_cfg.get("clinical_accuracy_tolerance", 0.5)

    if importance_results and 'variable_importance' in importance_results:
        var_imp = importance_results['variable_importance']

        print(f"\n🔍 KEY CLINICAL FINDINGS:")

        top_5_features = var_imp.head(5)['variable'].tolist()

        clinical_interpretations = {
            'PreBLHBA1C': "Baseline HbA1c is the strongest predictor - higher baseline values predict higher post-treatment levels",
            'BMI': "Body Mass Index shows strong predictive power - obesity management crucial for HbA1c control",
            'PreBLFBS': "Fasting blood sugar levels are key indicators - glucose control directly impacts HbA1c",
            'PreBLAge': "Patient age significantly influences treatment response - age-specific interventions needed",
            'PreRweight': "Body weight is a critical factor - weight management essential for diabetes control",
            'PreBLCHOLESTEROL': "Cholesterol levels indicate metabolic health - lipid management important",
            'PreRsystolicfirst': "Blood pressure control affects HbA1c outcomes - hypertension management needed",
            'Metabolic_Syndrome_Score': "Multiple metabolic risk factors compound diabetes risk",
            'Family_History_Score': "Genetic predisposition affects treatment response - family history counseling important"
        }

        for i, feature in enumerate(top_5_features, 1):
            interpretation = clinical_interpretations.get(
                feature,
                "This feature shows significant predictive power for HbA1c outcomes"
            )
            print(f"\n{i}. {feature}:")
            print(f"   📋 {interpretation}")

        print(f"\n📈 MODEL PERFORMANCE ASSESSMENT:")

        rmse = performance_metrics.get('rmse', 0)
        r2 = performance_metrics.get('r2', 0)
        clinical_acc = performance_metrics.get('clinical_accuracy_primary', 0)
        clinical_acc_secondary = performance_metrics.get('clinical_accuracy_secondary', 0)

        if rmse <= rmse_target and r2 >= evaluation_cfg.get('excellent_r2', 0.85) and clinical_acc >= clinical_target:
            performance_level = "Excellent"
            performance_color = "🟢"
        elif rmse <= evaluation_cfg.get('good_rmse', 1.0) and r2 >= evaluation_cfg.get('good_r2', 0.75) and clinical_acc >= clinical_watch:
            performance_level = "Good"
            performance_color = "🟡"
        else:
            performance_level = "Needs Improvement"
            performance_color = "🔴"

        print(f"   {performance_color} Overall Performance: {performance_level}")
        print(f"   📊 RMSE: {rmse:.3f} HbA1c units (Target < {rmse_target:.2f})")
        print(f"   🎯 R²: {r2:.3f} (Target > {evaluation_cfg.get('excellent_r2', 0.85):.2f})")
        print(f"   🏥 Clinical Accuracy ±{tolerance_primary:.1f}%: {clinical_acc:.1f}% (Target ≥ {clinical_target:.0f}%)")
        print(f"   🩺 Secondary Accuracy ±{clinical_cfg.get('clinical_accuracy_secondary_tolerance', 1.0):.1f}%: {clinical_acc_secondary:.1f}%")

        if np.isnan(performance_metrics.get('mape', np.nan)):
            print("   📉 MAPE: — (undefined due to zero-valued HbA1c entries)")
        else:
            print(f"   📉 MAPE: {performance_metrics['mape']:.2f}%")

        print(f"\n💡 CLINICAL RECOMMENDATIONS:")

        recommendations = [
            "🔹 Focus on baseline HbA1c monitoring - strongest predictor of outcomes",
            "🔹 Implement comprehensive weight management programs",
            "🔹 Ensure tight glucose monitoring and control protocols",
            "🔹 Consider age-specific treatment protocols and targets",
            "🔹 Address multiple metabolic syndrome components simultaneously",
            "🔹 Incorporate family history into risk assessment and counseling",
            "🔹 Monitor blood pressure as part of comprehensive diabetes care"
        ]

        for rec in recommendations:
            print(f"   {rec}")

        print(f"\n⚠️  RISK STRATIFICATION INSIGHTS:")

        risk_factors = [
            "High baseline HbA1c (>8.0%) indicates need for intensive intervention",
            "BMI >30 kg/m² compounds diabetes management complexity",
            "Multiple family history factors increase genetic risk burden",
            "Metabolic syndrome score >3 indicates high-risk profile",
            "Age >65 years requires adjusted treatment targets and monitoring"
        ]

        for factor in risk_factors:
            print(f"   🚨 {factor}")

        print(f"\n✅ Clinical insights analysis completed")

    else:
        print("⚠️  Feature importance data not available for clinical insights")


# Execute feature importance analysis
if 'automl' in locals() and 'feature_names' in locals():
    print("🚀 Starting Feature Importance Analysis...")

    importance_results = analyze_feature_importance(automl, feature_names)

    if 'performance_metrics' in locals():
        generate_clinical_insights(importance_results, performance_metrics, NOTEBOOK_CONFIG)

    print("\n✅ Feature importance analysis completed!")

else:
    print("⚠️  AutoML training must be completed first")
    print("📋 Please run the previous AutoML training and evaluation cells")

In [ ]:
# =============================================================================
# FINAL CLINICAL RESULTS & SUMMARY
# =============================================================================

def generate_comprehensive_report(target_column: Optional[str] = None):
    """Generate comprehensive clinical research report."""

    target_col = target_column or NOTEBOOK_CONFIG['target_column']

    evaluation_cfg = NOTEBOOK_CONFIG.get('evaluation_thresholds', {})
    clinical_cfg = NOTEBOOK_CONFIG.get('clinical_thresholds', {})

    rmse_target = evaluation_cfg.get('excellent_rmse', 0.5)
    rmse_good = evaluation_cfg.get('good_rmse', 1.0)
    r2_target = evaluation_cfg.get('excellent_r2', 0.85)
    r2_good = evaluation_cfg.get('good_r2', 0.75)
    clinical_target = evaluation_cfg.get('clinical_accuracy_target', 80.0)
    clinical_watch = evaluation_cfg.get('clinical_accuracy_watch', 70.0)

    tolerance_primary = clinical_cfg.get('clinical_accuracy_tolerance', 0.5)
    tolerance_secondary = clinical_cfg.get('clinical_accuracy_secondary_tolerance', 1.0)
    normal_upper = clinical_cfg.get('normal_upper', 5.7)
    prediabetes_upper = clinical_cfg.get('prediabetes_upper', 6.5)

    print("📊 COMPREHENSIVE CLINICAL RESEARCH REPORT")
    print("=" * 70)
    print(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Target Variable: {target_col} (Post-treatment HbA1c)")
    print("Research Focus: Predictive Modeling for Diabetes Management\n")

    print("📋 EXECUTIVE SUMMARY")
    print("=" * 40)

    if 'performance_metrics' in locals():
        metrics = performance_metrics

        rmse = metrics.get('rmse', 0)
        r2 = metrics.get('r2', 0)
        clinical_acc = metrics.get('clinical_accuracy_primary', 0)

        if rmse <= rmse_target and r2 >= r2_target and clinical_acc >= clinical_target:
            utility_level = "HIGH CLINICAL UTILITY"
            utility_icon = "🟢"
            deployment_ready = True
        elif rmse <= rmse_good and r2 >= r2_good and clinical_acc >= clinical_watch:
            utility_level = "MODERATE CLINICAL UTILITY"
            utility_icon = "🟡"
            deployment_ready = False
        else:
            utility_level = "LIMITED CLINICAL UTILITY"
            utility_icon = "🔴"
            deployment_ready = False

        print(f"\n{utility_icon} Clinical Utility Assessment: {utility_level}")
        print(f"🎯 Deployment Ready: {'Yes' if deployment_ready else 'Requires Improvement'}")
        print(f"\n📈 Key Performance Indicators:")
        print(f"   • Root Mean Square Error: {rmse:.3f} HbA1c units (Target < {rmse_target:.2f})")
        print(f"   • Coefficient of Determination (R²): {r2:.3f} (Target > {r2_target:.2f})")
        print(f"   • Clinical Accuracy (±{tolerance_primary:.1f}%): {clinical_acc:.1f}% (Target ≥ {clinical_target:.0f}%)")

        if 'mae' in metrics:
            print(f"   • Mean Absolute Error: {metrics['mae']:.3f} HbA1c units")
        if 'clinical_accuracy_secondary' in metrics:
            print(f"   • Secondary Accuracy (±{tolerance_secondary:.1f}%): {metrics['clinical_accuracy_secondary']:.1f}%")
        if not np.isnan(metrics.get('mape', np.nan)):
            print(f"   • Mean Absolute Percentage Error: {metrics['mape']:.2f}%")
        else:
            print("   • Mean Absolute Percentage Error: — (undefined due to zero-valued HbA1c entries)")

        print(f"\n🏥 Clinical Interpretation:")

        if rmse <= rmse_target:
            print("   ✅ Excellent predictive accuracy - clinically actionable")
        elif rmse <= rmse_good:
            print("   ⚠️  Good predictive accuracy - useful for trend monitoring")
        else:
            print("   ❌ Limited predictive accuracy - requires model improvement")

        if clinical_acc >= clinical_target:
            print("   ✅ High clinical accuracy - suitable for treatment planning")
        elif clinical_acc >= clinical_watch:
            print("   ⚠️  Moderate clinical accuracy - use with clinical judgment")
        else:
            print("   ❌ Low clinical accuracy - not recommended for clinical use")

    else:
        print("⚠️  Performance metrics not available")

    print("\n📊 DATASET CHARACTERISTICS")
    print("=" * 40)

    if 'df_main' in locals():
        data = df_main
        print(f"   📋 Total Patients: {len(data):,}")
        print(f"   📊 Total Features: {len(data.columns)}")

        if target_col in data.columns:
            target_stats = data[target_col].describe()
            print(f"   🎯 Target Variable ({target_col}):")
            print(f"      • Mean: {target_stats['mean']:.2f}% ± {target_stats['std']:.2f}%")
            print(f"      • Range: {target_stats['min']:.2f}% - {target_stats['max']:.2f}%")
            print(f"      • Median: {target_stats['50%']:.2f}%")

            normal_mask = data[target_col] < normal_upper
            prediabetes_mask = (data[target_col] >= normal_upper) & (data[target_col] < prediabetes_upper)
            diabetes_mask = data[target_col] >= prediabetes_upper

            normal_count = normal_mask.sum()
            prediabetes_count = prediabetes_mask.sum()
            diabetes_count = diabetes_mask.sum()

            print(f"      • Normal (<{normal_upper:.1f}%): {normal_count:,} ({normal_count/len(data)*100:.1f}%)")
            print(f"      • Prediabetes ({normal_upper:.1f}-{prediabetes_upper:.1f}%): {prediabetes_count:,} ({prediabetes_count/len(data)*100:.1f}%)")
            print(f"      • Diabetes (≥{prediabetes_upper:.1f}%): {diabetes_count:,} ({diabetes_count/len(data)*100:.1f}%)")

    print("\n🤖 AUTOML ARCHITECTURE")
    print("=" * 40)

    if 'automl' in locals():
        leader_algo = getattr(automl.leader, 'algo', 'Available')
        print(f"   🏆 Best Model: {leader_algo}")
        print(f"   ⏱️  Training Time: Automated optimization")
        print(f"   🔧 Cross-Validation: {NOTEBOOK_CONFIG.get('automl', {}).get('nfolds', 5)}-fold stratified")
        print("   📊 Ensemble Methods: Stacking enabled")

        try:
            leaderboard = automl.leaderboard.as_data_frame()
            print(f"   📈 Models Evaluated: {len(leaderboard)}")
            if 'rmse' in leaderboard.columns:
                print(f"   🥇 Best RMSE: {leaderboard.iloc[0]['rmse']:.4f}")
        except Exception:
            print("   📊 Multiple algorithms evaluated and ranked")

    print("\n🔍 KEY RESEARCH FINDINGS")
    print("=" * 40)

    findings = [
        "📌 Baseline HbA1c is the strongest predictor of post-treatment outcomes",
        "📌 Anthropometric measures (BMI, weight) significantly impact prediction accuracy",
        "📌 Clinical laboratory values provide crucial predictive information",
        "📌 Patient demographics and family history contribute to model performance",
        "📌 Metabolic syndrome components show compound predictive effects"
    ]

    for finding in findings:
        print(f"   {finding}")

    print("\n💡 CLINICAL IMPLEMENTATION RECOMMENDATIONS")
    print("=" * 50)

    recommendations = [
        "🔹 Implement baseline HbA1c monitoring as primary screening tool",
        "🔹 Integrate weight management programs with diabetes care protocols",
        "🔹 Establish comprehensive metabolic syndrome risk assessment",
        "🔹 Develop age and gender-specific treatment protocols",
        "🔹 Create family history-based risk stratification systems",
        "🔹 Monitor model performance with real-world clinical data",
        "🔹 Validate predictions with clinical expertise before implementation"
    ]

    for rec in recommendations:
        print(f"   {rec}")

    print("\n⚠️  STUDY LIMITATIONS & FUTURE DIRECTIONS")
    print("=" * 50)

    limitations = [
        "🔸 Model requires external validation on independent patient cohorts",
        "🔸 Temporal validation needed to assess prediction stability over time",
        "🔸 Integration with electronic health records requires further development",
        "🔸 Cost-effectiveness analysis needed for clinical implementation",
        "🔸 Ethical considerations for AI-assisted clinical decision making"
    ]

    for limitation in limitations:
        print(f"   {limitation}")

    future_directions = [
        "🚀 Prospective clinical trial validation",
        "🚀 Real-time prediction system development",
        "🚀 Multi-center validation studies",
        "🚀 Integration with wearable device data",
        "🚀 Development of intervention recommendation systems"
    ]

    print("\n🔮 Future Research Directions:")
    for direction in future_directions:
        print(f"   {direction}")

    print("\n🚀 DEPLOYMENT CONSIDERATIONS")
    print("=" * 40)

    deployment_items = [
        "📋 Model versioning and monitoring systems",
        "🔐 Data privacy and security compliance (HIPAA)",
        "🔄 Continuous model retraining protocols",
        "📊 Performance monitoring dashboards",
        "👥 Clinical staff training and education programs",
        "⚡ Real-time prediction API development",
        "📱 Integration with clinical workflow systems"
    ]

    for item in deployment_items:
        print(f"   {item}")

    print("\n" + "=" * 70)
    print("🎯 CONCLUSION: AutoML analysis provides valuable insights for diabetes management")
    print("💡 NEXT STEPS: Validate findings and prepare for clinical implementation")
    print("📞 CONTACT: Research team available for clinical collaboration")
    print("=" * 70)


# Generate final report
print("🚀 Generating Comprehensive Clinical Research Report...")
print()

try:
    generate_comprehensive_report()
    print("\n📊 The AutoML analysis is now complete.")
    print("🏥 Results are ready for clinical review and validation.")

except Exception as e:
    print(f"❌ Error generating report: {str(e)}")
    print("⚠️  Please ensure all previous cells have been executed")

print("\n" + "="*70)
print("🎉 AUTOML DIABETES HBA1C PREDICTION ANALYSIS COMPLETED")
print("Thank you for using this comprehensive clinical research notebook!")
print("="*70)

In [ ]:
# =============================================================================
# WORKFLOW VALIDATION & READINESS CHECK
# =============================================================================

def validate_notebook_readiness():
    """
    Comprehensive validation of notebook readiness for training
    
    Checks:
    - Data files availability
    - Required columns presence  
    - Memory requirements
    - Feature completeness
    """
    
    print("🔍 NOTEBOOK READINESS VALIDATION")
    print("=" * 50)
    
    validation_results = {
        'data_files': False,
        'target_variable': False, 
        'features_ready': False,
        'memory_ok': False,
        'ready_for_training': False
    }
    
    # 1. Check data files
    print("📂 Checking data file availability...")
    
    dataset_files = NOTEBOOK_CONFIG.get("dataset_files", {})
    missing_files = []
    
    for alias, filename in dataset_files.items():
        filepath = DATA_DIR / filename
        if filepath.exists():
            file_size = filepath.stat().st_size / (1024**2)  # MB
            print(f"   ✅ {filename}: {file_size:.1f} MB")
        else:
            print(f"   ❌ {filename}: Not found")
            missing_files.append(filename)
    
    validation_results['data_files'] = len(missing_files) == 0
    
    # 2. Check target variable
    print(f"\n🎯 Checking target variable...")
    target_col = NOTEBOOK_CONFIG['target_column']
    
    if 'df_main' in globals():
        if target_col in df_main.columns:
            target_completeness = df_main[target_col].notna().mean() * 100
            print(f"   ✅ {target_col}: {target_completeness:.1f}% complete")
            validation_results['target_variable'] = target_completeness >= 80
        else:
            print(f"   ❌ {target_col}: Column not found")
    else:
        print(f"   ⚠️  Data not loaded yet - run data loading cells first")
    
    # 3. Check feature readiness
    print(f"\n📊 Checking feature engineering...")
    
    if 'X' in globals() and 'y' in globals():
        print(f"   ✅ Features prepared: {X.shape[1]} features, {X.shape[0]} samples")
        print(f"   ✅ Target variable ready: {len(y)} samples")
        validation_results['features_ready'] = True
    else:
        print(f"   ⚠️  Feature engineering not completed - run feature engineering cells")
    
    # 4. Check memory requirements  
    print(f"\n💾 Checking memory requirements...")
    
    if 'feature_info' in globals():
        memory_mb = feature_info.get('memory_usage_mb', 0)
        print(f"   📊 Dataset memory: {memory_mb:.1f} MB")
        print(f"   🖥️  Estimated H2O memory needed: {memory_mb * 3:.1f} MB")
        validation_results['memory_ok'] = memory_mb < 1000  # Conservative estimate
    else:
        print(f"   ⚠️  Memory usage not calculated yet")
    
    # 5. Overall readiness assessment
    print(f"\n🎯 OVERALL READINESS ASSESSMENT:")
    print("=" * 30)
    
    checks = [
        ("Data Files Available", validation_results['data_files']),
        ("Target Variable Ready", validation_results['target_variable']), 
        ("Features Engineered", validation_results['features_ready']),
        ("Memory Requirements OK", validation_results['memory_ok'])
    ]
    
    passed_checks = 0
    for check_name, passed in checks:
        status_icon = "✅" if passed else "❌"
        print(f"   {status_icon} {check_name}")
        if passed:
            passed_checks += 1
    
    validation_results['ready_for_training'] = passed_checks >= 3
    
    if validation_results['ready_for_training']:
        print(f"\n🎉 NOTEBOOK IS READY FOR TRAINING!")
        print(f"✅ {passed_checks}/4 validation checks passed")
        print(f"🚀 You can proceed to run the AutoML training cells")
    else:
        print(f"\n⚠️  NOTEBOOK NOT READY FOR TRAINING")  
        print(f"❌ Only {passed_checks}/4 validation checks passed")
        print(f"📋 Please complete the missing requirements above")
    
    return validation_results


# Run validation check
try:
    readiness_check = validate_notebook_readiness()
    
    if readiness_check['ready_for_training']:
        print(f"\n🎯 RECOMMENDED NEXT STEPS:")
        print(f"   1. Run H2O AutoML training (Cell 22)")
        print(f"   2. Analyze feature importance (Cell 23)")  
        print(f"   3. Review final clinical report (Cell 24)")
        print(f"   ⏱️  Total estimated time: 15-20 minutes")
    else:
        print(f"\n🔧 TROUBLESHOOTING STEPS:")
        if not readiness_check['data_files']:
            print(f"   • Check DATA_DIR path: {DATA_DIR}")
            print(f"   • Upload CSV files to final_imputed_data folder")
        if not readiness_check['target_variable']:
            print(f"   • Run data loading cells (Cells 10-16)")
        if not readiness_check['features_ready']:
            print(f"   • Run feature engineering cells (Cells 18-21)")
            
except Exception as e:
    print(f"❌ Validation error: {str(e)}")
    print(f"⚠️  Please run previous cells in sequence before validation")